In [1]:
import os # for handling file paths
import pandas as pd  # data manipulation
import holidays # for handling holidays
from astral.sun import dawn, dusk  # for calculating sunrise/sunset times
from astral import LocationInfo # for defining location for sunrise/sunset calculations

In [2]:
cwd = os.getcwd()

parent_dir = os.path.dirname(cwd)
data_dir = os.path.join(parent_dir, 'data/raw')
household_data_path = os.path.join(data_dir, 'opsd-household_data-2020-04-15/household_data_60min_singleindex.csv')
household_data = pd.read_csv(household_data_path, index_col=0, parse_dates=True)

household_data.columns = household_data.columns.str.replace('DE_KN_', '', regex=False)
residential4_data = household_data.filter(regex='^residential4_')
residential4_data.columns = residential4_data.columns.str.replace('residential4_', '', regex=False)
start_date = pd.Timestamp('2015-10-13 16:00:00', tz='UTC')
end_date = pd.Timestamp('2017-03-06 14:00:00', tz='UTC')
residential4_data = residential4_data[['grid_import', 'grid_export', 'pv']]
residential4_data = residential4_data.loc[start_date:end_date]
residential4_data = residential4_data.diff().iloc[1:]
residential4_data['energy_demand'] = residential4_data['grid_import'] + residential4_data['pv'] - residential4_data['grid_export']

residential4_data.head(3)

,grid_import,grid_export,pv,energy_demand
utc_timestamp,,,,
2015-10-13 17:00:00+00:00,1.248,0.0,0.0,1.248
2015-10-13 18:00:00+00:00,0.765,0.0,0.0,0.765
2015-10-13 19:00:00+00:00,0.701,0.0,0.0,0.701


In [3]:
data = residential4_data[['energy_demand']].copy()

In [4]:
de_holidays = holidays.CountryHoliday(
    'DE',
    subdiv='BW',   # Baden-Württemberg
    years=range(2015, 2018)
)
holiday_dates = pd.to_datetime(list(de_holidays.keys()))
holiday_dates

DatetimeIndex(['2016-01-01', '2016-03-25', '2016-03-28', '2016-05-01',
               '2016-05-05', '2016-05-16', '2016-10-03', '2016-12-25',
               '2016-12-26', '2016-01-06', '2016-05-26', '2016-11-01',
               '2017-01-01', '2017-04-14', '2017-04-17', '2017-05-01',
               '2017-05-25', '2017-06-05', '2017-10-03', '2017-12-25',
               '2017-12-26', '2017-10-31', '2017-01-06', '2017-06-15',
               '2017-11-01', '2015-01-01', '2015-04-03', '2015-04-06',
               '2015-05-01', '2015-05-14', '2015-05-25', '2015-10-03',
               '2015-12-25', '2015-12-26', '2015-01-06', '2015-06-04',
               '2015-11-01'],
              dtype='datetime64[s]', freq=None)

In [5]:
data['is_holiday_or_weekend'] = data.index.to_series().apply(lambda x: x in holiday_dates or x.weekday() >= 5)
data.head(3)

,energy_demand,is_holiday_or_weekend
utc_timestamp,,
2015-10-13 17:00:00+00:00,1.248,False
2015-10-13 18:00:00+00:00,0.765,False
2015-10-13 19:00:00+00:00,0.701,False


In [6]:
city = LocationInfo("Konstanz", "Germany")
data['daylight_flag'] = data.index.to_series().apply(lambda x: dawn(city.observer, date=x.date()) <= x <= dusk(city.observer, date=x.date()))
data.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag
utc_timestamp,,,
2015-10-13 17:00:00+00:00,1.248,False,True
2015-10-13 18:00:00+00:00,0.765,False,False
2015-10-13 19:00:00+00:00,0.701,False,False


In [ ]:
def time_of_day(date): # has to be converted to the numerical format to be used in the model
    hour = date.hour
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    else:
        return 'evening'
    
data['time_of_day'] = data.index.to_series().apply(time_of_day)
data.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag,time_of_day
utc_timestamp,,,,
2015-10-13 17:00:00+00:00,1.248,False,True,afternoon
2015-10-13 18:00:00+00:00,0.765,False,False,evening
2015-10-13 19:00:00+00:00,0.701,False,False,evening


In [8]:
def get_season(date):  # has to be converted to the numerical format to be used in the model
    month = date.month
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'autumn'
data['season'] = data.index.to_series().apply(get_season)
data.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag,time_of_day,season
utc_timestamp,,,,,
2015-10-13 17:00:00+00:00,1.248,False,True,afternoon,autumn
2015-10-13 18:00:00+00:00,0.765,False,False,evening,autumn
2015-10-13 19:00:00+00:00,0.701,False,False,evening,autumn


In [9]:
# save the processed data
processed_data_dir = os.path.join(parent_dir, 'data/processed')
processed_data_path = os.path.join(processed_data_dir, 'residential4_energy_demand.csv')
data.to_csv(processed_data_path)

In [10]:
# read newly saved data to verify
verified_data = pd.read_csv(processed_data_path, index_col=0, parse_dates=True)
verified_data.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag,time_of_day,season
utc_timestamp,,,,,
2015-10-13 17:00:00+00:00,1.248,False,True,afternoon,autumn
2015-10-13 18:00:00+00:00,0.765,False,False,evening,autumn
2015-10-13 19:00:00+00:00,0.701,False,False,evening,autumn
